# 11과 - 에이전트 간 (A2A) 프로토콜


## 설정


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## What is the A2A Protocol?

The **Agent-to-Agent (A2A) protocol** is an open standard that enables AI agents to communicate,
discover each other, and collaborate — even when they are built on different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agents publish an *Agent Card* that describes their capabilities, making it
  easy for other agents (or orchestrators) to find the right specialist for a task.
- **Message Passing** – Agents exchange structured messages through a common protocol, so a
  request from one agent can be understood and fulfilled by another regardless of its internal
  implementation.
- **Task Lifecycle** – A2A defines states such as *submitted*, *working*, *completed*, and
  *failed*, giving the orchestrator full visibility into how a delegated task is progressing.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## 전문 여행사 만들기


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## 워크플로우를 통한 다중 에이전트 협업

세 가지 에이전트를 A2A 메시지 전달을 반영한 순차적 워크플로우로 연결합니다:

1. <strong>CurrencyExchangeAgent</strong>가 사용자 요청을 받아 통화 안내를 생성합니다.
2. <strong>ActivityPlannerAgent</strong>가 강화된 컨텍스트를 받아 활동 추천을 추가합니다.
3. <strong>TravelManagerAgent</strong>가 두 입력을 종합해 최종 여행 요약을 만듭니다.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 프로덕션에서 A2A 이해하기

프로덕션 환경에서 A2A 프로토콜은 강력한 크로스 서비스 시나리오를 가능하게 합니다:

| 기능 | 설명 |
|---|---|
| **크로스 프레임워크 상호 운용** | 한 프레임워크로 구축된 에이전트가 다른 모든 A2A 호환 프레임워크로 구축된 에이전트에게 작업을 위임할 수 있어 진정한 조직 간 상호 운용성을 가능하게 합니다. |
| **서비스 경계** | 에이전트는 별도의 마이크로서비스, 클라우드 지역 또는 심지어 다른 조직에 존재하면서도 원활하게 협업할 수 있습니다. |
| **동적 검색** | 오케스트레이터는 런타임에 에이전트 카드 레지스트리를 조회하여 특정 하위 작업에 가장 적합한 전문가를 찾을 수 있습니다. |
| **스트리밍 & 푸시 알림** | A2A는 실시간 진행 상황 업데이트를 위해 서버 전송 이벤트(SSE)를 지원하며, 장기 실행 작업에 대해 푸시 알림을 지원합니다. |

위에서 구축한 워크플로우는 이 패턴의 단순화된 인프로세스 버전입니다. 실제
배포에서는 각 에이전트가 HTTP 엔드포인트를 노출하고, 에이전트 카드를 게시하며,
A2A JSON-RPC 프로토콜을 통해 통신할 것입니다.


## 요약

이 강의에서 배운 내용:

1. **A2A 프로토콜이 무엇인지** — 에이전트 간 발견, 메시징 및 작업 관리를 위한 개방형 표준.
   .
2. **특수화된 에이전트 생성 방법** — 통화 환전 에이전트, 활동 계획 에이전트,
   여행 관리자 조정자(agent).
3. **에이전트 간의 워크플로우 구성 방법** — `WorkflowBuilder`를 사용하여
   에이전트 간 순차적 메시지 전달 모델링.
4. **A2A가 실제 환경에서 작동하는 방법** — 동적 발견 및 스트리밍 업데이트를 통한
   프레임워크 및 서비스 간 협업 구현.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
